# Exercises: weighted metapaths on the HALD knowledge graph

These exercises accompany the notebook **Predicting `Gene –reduce→ Disease` links in the HALD knowledge graph with weighted metapaths**. They take about **30 minutes** in total, and each one practises an idea from a different part of the main notebook.

| Exercise | What you practise | Main notebook | Time |
|---|---|---|---|
| 1 | What the damping exponent $w$ does | Sections 5 and 7 | 7 min |
| 2 | Computing a DWPC by hand, then checking it | Sections 3 and 5 | 8 min |
| 3 | Drawing an ROC curve by hand | Section 11 | 8 min |
| 4 | Testing the method against fairer negative examples | Sections 6 and 10 | 7 min |

### How to work
* Run the **Setup** section first. It takes about a minute, so read Exercise 1 while it runs.
* Each exercise says *what you are asked to do*, then gives a **by hand** part (use paper) and a **code** part.
* In a code cell, a line that ends with `...` followed by a `# TODO` comment is **yours to write**. Everything else is already written; read it first, because it shows the style and the names to use.
* After each exercise there are questions. Write your answer in a sentence or two before moving on.
* Random seeds are fixed, so the numbers you get will be the same as the ones in the main notebook and in the solutions.

## Setup
The exercises reuse the variables and functions of the main notebook. The cells below are the code cells of Sections 1 to 8 of the **networkx version** of the notebook, copied without change, so no database is needed. They read `HALD.rdf` (keep it in the same folder as this notebook), build the network, find the metapaths, define the DWPC functions, build the data set of positive and negative pairs, and compute the features.

Two values that the main notebook *chooses* are written in directly, so that the exercises do not have to repeat the choosing: the damping exponent `best_w = 0.2` (Section 7) and the regularisation strength `best_C = 0.00003` (Section 10). Exercise 1 will repeat a small part of the choice of $w$ so that you can see how it works.

In [1]:
import networkx as nx
import pandas as pd
from rdflib import Graph, RDF, OWL

HALD_RDF_FILE = "./HALD_Dataset/HALD.rdf"                              # path to the HALD RDF/XML file
HALD_NAMESPACE = "http://www.dummy.info/new.owl#"         # the web address that prefixes every HALD name

def read_hald_rdf(rdf_file, namespace):
    # Returns node_class (node name -> class name) and predicates_of_edge ((source, target) -> set of predicates).
    rdf_graph = Graph()
    rdf_graph.parse(rdf_file, format="xml")
    is_hald_name = lambda uri: str(uri).startswith(namespace)
    short_name = lambda uri: str(uri)[len(namespace):]

    node_class = {}
    for class_uri in rdf_graph.subjects(RDF.type, OWL.Class):
        if is_hald_name(class_uri):
            for node_uri in rdf_graph.subjects(RDF.type, class_uri):
                node_class.setdefault(short_name(node_uri), short_name(class_uri))

    relationship_uris = {uri for uri in rdf_graph.subjects(RDF.type, OWL.ObjectProperty) if is_hald_name(uri)}
    predicates_of_edge = {}
    for source_uri, predicate_uri, target_uri in rdf_graph:
        if predicate_uri in relationship_uris and source_uri != target_uri:
            source, target = short_name(source_uri), short_name(target_uri)
            if source in node_class and target in node_class:
                predicates_of_edge.setdefault((source, target), set()).add(short_name(predicate_uri))
    return node_class, predicates_of_edge

In [2]:
node_class, predicates_of_edge = read_hald_rdf(HALD_RDF_FILE, HALD_NAMESPACE)
print(len(node_class), "nodes,", len(predicates_of_edge), "node pairs joined by at least one predicate")

12256 nodes, 59692 node pairs joined by at least one predicate


In [3]:
nodes_by_class = dict()
for name, class_name in node_class.items():
    nodes_by_class.setdefault(class_name, []).append(name)
nodes_by_class = {class_name: sorted(names) for class_name, names in nodes_by_class.items()}

total_nodes = sum(len(names) for names in nodes_by_class.values())
print(f"{len(nodes_by_class)} classes, {total_nodes:,} nodes  (the full HALD file has 10 classes and 12,256 nodes)")
print("first genes:", nodes_by_class["Gene"][:5])
print("first diseases:", nodes_by_class["Disease"][:5])

10 classes, 12,256 nodes  (the full HALD file has 10 classes and 12,256 nodes)
first genes: ['A1BG', 'A2M', 'A2ML1', 'AAK1', 'ABAT']
first diseases: ['ACTH_Secreting_Pituitary_Adenoma', 'AIDS_Arteritis_Central_Nervous_System', 'AIDS_Associated_Nephropathy', 'AIDS_Dementia_Complex', 'AIDS_Related_Complex']


In [4]:
target_pairs = sorted((source, target) for (source, target), predicates in predicates_of_edge.items()
                      if node_class[source] == "Gene" and node_class[target] == "Disease" and "reduce" in predicates)

print(len(target_pairs), "known Gene-reduce-Disease pairs")
print("examples:", target_pairs[:3])

132 known Gene-reduce-Disease pairs
examples: [('ABCC3', 'Pancreatic_Neoplasms'), ('ACE', 'Anemia_Sickle_Cell'), ('ACE', 'Cardiovascular_Diseases')]


In [5]:
hald_network = nx.DiGraph()
for name, class_name in node_class.items():
    hald_network.add_node(name, node_class=class_name)

for (source, target), predicates in predicates_of_edge.items():
    if node_class[source] == "Gene" and node_class[target] == "Disease":
        predicates = predicates - {"reduce"}
    if predicates:                                       # some predicate remains, so the edge stays
        hald_network.add_edge(source, target, predicates=predicates)

print(hald_network.number_of_nodes(), "nodes,", hald_network.number_of_edges(), "edges")

12256 nodes, 59638 edges


In [6]:
edges_by_metaedge = dict()
for source, target in hald_network.edges:
    metaedge = (node_class[source], node_class[target])
    # if the dictionary entry does not exist, create an empty list, then append to it.
    edges_by_metaedge.setdefault(metaedge, []).append((source, target))

# sort everything, so that the results do not depend on the order in which the edges happened to be read
edges_by_metaedge = {metaedge: sorted(edges) for metaedge, edges in sorted(edges_by_metaedge.items())}

number_of_edges = sum(len(edges) for edges in edges_by_metaedge.values())
print(f"{len(edges_by_metaedge)} metaedges, {number_of_edges:,} distinct edges  (the full HALD file has 63 metaedges and 59,638 edges)")

metaedge_sizes = pd.DataFrame(
    [(source_class, target_class, len(edges)) for (source_class, target_class), edges in edges_by_metaedge.items()],
    columns=["source class", "target class", "number of edges"]).sort_values("number of edges", ascending=False)
metaedge_sizes.head(10)

63 metaedges, 59,638 distinct edges  (the full HALD file has 63 metaedges and 59,638 edges)


,source class,target class,number of edges
9,Disease,Disease,36724
19,Gene,Disease,7498
10,Disease,Gene,4559
20,Gene,Gene,3266
28,Lipid,Disease,963
1,Carbohydrate,Disease,894
11,Disease,Lipid,695
8,Disease,Carbohydrate,681
35,Mutation,Disease,570
12,Disease,Mutation,424


In [7]:
from collections import Counter

out_degree = {}     # out_degree[metaedge][node] = number of edges leaving the node in that metaedge
in_degree = {}      # in_degree[metaedge][node]  = number of edges arriving at the node in that metaedge

for metaedge, edges in edges_by_metaedge.items():
    out_degree[metaedge] = Counter(source for source, target in edges)
    in_degree[metaedge] = Counter(target for source, target in edges)

print("example: out-degree of INS in (Gene, Disease):", out_degree[("Gene", "Disease")]["INS"])
print("example: in-degree of Diabetes_Mellitus in (Gene, Disease):", in_degree[("Gene", "Disease")]["Diabetes_Mellitus"])
print("a gene with no edges in the metaedge gets 0:", out_degree[("Gene", "Disease")]["A1BG"])

example: out-degree of INS in (Gene, Disease): 168
example: in-degree of Diabetes_Mellitus in (Gene, Disease): 135
a gene with no edges in the metaedge gets 0: 0


In [8]:
MAX_LENGTH = 3          # longest metapath we consider, counted in steps (edges)

steps_from_class = {}
for source_class, target_class in edges_by_metaedge:
    metaedge = (source_class, target_class)
    steps_from_class.setdefault(source_class, []).append((">", target_class, metaedge))    # forward step
    steps_from_class.setdefault(target_class, []).append(("<", source_class, metaedge))    # backward step

for direction, arrive_at, metaedge in steps_from_class["Gene"][:4]:
    print(f"from Gene, step '{direction}' to {arrive_at:12s} using metaedge {metaedge}")

from Gene, step '<' to Carbohydrate using metaedge ('Carbohydrate', 'Gene')
from Gene, step '<' to Disease      using metaedge ('Disease', 'Gene')
from Gene, step '>' to Carbohydrate using metaedge ('Gene', 'Carbohydrate')
from Gene, step '>' to Disease      using metaedge ('Gene', 'Disease')


In [9]:
def find_metapaths(steps_from_class, max_length, start_class, end_class):
    # Returns a dictionary: metapath name -> list of steps; each step is (direction, class arrived at, metaedge used)
    found = {}

    def extend(steps_so_far, current_class):
        if steps_so_far and current_class == end_class:
            name = start_class + "".join(direction + arrive_at for direction, arrive_at, metaedge in steps_so_far)
            found[name] = steps_so_far
        if len(steps_so_far) == max_length:
            return
        for step in steps_from_class.get(current_class, []):
            direction, arrive_at, metaedge = step
            extend(steps_so_far + [step], arrive_at)

    extend([], start_class)
    return found

metapaths = find_metapaths(steps_from_class, MAX_LENGTH, "Gene", "Disease")
print(len(metapaths), f"metapaths from Gene to Disease with at most {MAX_LENGTH} steps")
for i in metapaths:
    print(i)

516 metapaths from Gene to Disease with at most 3 steps
Gene<Carbohydrate>Carbohydrate>Disease
Gene<Carbohydrate>Carbohydrate<Disease
Gene<Carbohydrate<Carbohydrate>Disease
Gene<Carbohydrate<Carbohydrate<Disease
Gene<Carbohydrate>Disease
Gene<Carbohydrate>Disease>Disease
Gene<Carbohydrate>Disease<Disease
Gene<Carbohydrate>Gene<Disease
Gene<Carbohydrate>Gene>Disease
Gene<Carbohydrate>Lipid<Disease
Gene<Carbohydrate>Lipid>Disease
Gene<Carbohydrate>Mutation<Disease
Gene<Carbohydrate>Mutation>Disease
Gene<Carbohydrate>Peptide<Disease
Gene<Carbohydrate>Peptide>Disease
Gene<Carbohydrate>Protein<Disease
Gene<Carbohydrate>Protein>Disease
Gene<Carbohydrate>RNA<Disease
Gene<Carbohydrate>RNA>Disease
Gene<Carbohydrate<Disease
Gene<Carbohydrate<Disease>Disease
Gene<Carbohydrate<Disease<Disease
Gene<Carbohydrate<Gene<Disease
Gene<Carbohydrate<Gene>Disease
Gene<Carbohydrate<Lipid<Disease
Gene<Carbohydrate<Lipid>Disease
Gene<Carbohydrate<Mutation<Disease
Gene<Carbohydrate<Mutation>Disease
Gene<Carbohy

In [10]:
import numpy as np
import scipy.sparse as sp

node_position = {class_name: {node: position for position, node in enumerate(names)}
                 for class_name, names in nodes_by_class.items()}

adjacency_matrices = {}
for (source_class, target_class), edges in edges_by_metaedge.items():
    rows = [node_position[source_class][source] for source, target in edges]
    columns = [node_position[target_class][target] for source, target in edges]
    shape = (len(nodes_by_class[source_class]), len(nodes_by_class[target_class]))
    adjacency_matrices[(source_class, target_class)] = sp.csr_matrix((np.ones(len(edges)), (rows, columns)), shape=shape)

gene_disease_matrix = adjacency_matrices[("Gene", "Disease")]
print("the (Gene, Disease) matrix has shape", gene_disease_matrix.shape, "and", gene_disease_matrix.nnz, "edges")

the (Gene, Disease) matrix has shape (5624, 3501) and 7498 edges


In [11]:
def weighted_adjacency_matrices(w, adjacency_matrices, out_degree, in_degree, nodes_by_class):
    # the weighted matrix W of every metaedge, for damping exponent w
    weighted = {}
    for metaedge, adjacency in adjacency_matrices.items():
        source_class, target_class = metaedge
        out_degrees = np.array([out_degree[metaedge][node] for node in nodes_by_class[source_class]], dtype=float)
        in_degrees = np.array([in_degree[metaedge][node] for node in nodes_by_class[target_class]], dtype=float)
        out_weights = np.where(out_degrees > 0, out_degrees, 1.0) ** -w      # d_out ^ -w  (0 replaced by 1)
        in_weights = np.where(in_degrees > 0, in_degrees, 1.0) ** -w         # d_in  ^ -w
        weighted[metaedge] = (sp.diags(out_weights) @ adjacency @ sp.diags(in_weights)).tocsr()
    return weighted

example_w = 0.4          # the value found best by the original papers; we choose our own w in Section 7
weighted_matrices = weighted_adjacency_matrices(example_w, adjacency_matrices, out_degree, in_degree, nodes_by_class)

# one entry by hand: INS -> Diabetes_Mellitus
gene, disease = "INS", "Diabetes_Mellitus"
metaedge = ("Gene", "Disease")
by_hand = (out_degree[metaedge][gene] * in_degree[metaedge][disease]) ** -example_w
from_matrix = weighted_matrices[metaedge][node_position["Gene"][gene], node_position["Disease"][disease]]
print(f"weight of the edge {gene} -> {disease}: by hand {by_hand:.5f}, from the matrix {from_matrix:.5f}")

weight of the edge INS -> Diabetes_Mellitus: by hand 0.01810, from the matrix 0.01810


In [12]:
def step_matrices_of(steps, weighted_matrices):
    # the matrix of every step of the metapath (transposed for backward steps)
    matrices = []
    for direction, arrive_at, metaedge in steps:
        weighted = weighted_matrices[metaedge]
        matrices.append(weighted if direction == ">" else weighted.T.tocsr())
    return matrices

def walk_matrix(steps, weighted_matrices):
    matrices = step_matrices_of(steps, weighted_matrices)
    result = matrices[0]
    for matrix in matrices[1:]:
        result = result @ matrix
    return result

walks = walk_matrix(metapaths["Gene>Gene>Disease"], weighted_matrices)
print("Gene>Gene>Disease: matrix of shape", walks.shape, "with", walks.nnz, "gene-disease pairs joined by at least one route")

Gene>Gene>Disease: matrix of shape (5624, 3501) with 53260 gene-disease pairs joined by at least one route


In [13]:
def dwpc_matrix(steps, weighted_matrices):
    # DWPC of every (gene, disease) pair for one metapath: rows = genes, columns = diseases
    assert len(steps) <= 3, "the matrix method is only corrected for metapaths of length up to 3"
    result = walk_matrix(steps, weighted_matrices)
    if len(steps) == 3:
        M1, M2, M3 = step_matrices_of(steps, weighted_matrices)
        classes = ["Gene"] + [arrive_at for direction, arrive_at, metaedge in steps]
        n0_can_equal_n2 = classes[0] == classes[2]
        n1_can_equal_n3 = classes[1] == classes[3]
        if n0_can_equal_n2:
            result = result - sp.diags(np.asarray(M1.multiply(M2.T).sum(axis=1)).ravel()) @ M3
        if n1_can_equal_n3:
            result = result - M1 @ sp.diags(np.asarray(M2.multiply(M3.T).sum(axis=1)).ravel())
        if n0_can_equal_n2 and n1_can_equal_n3:
            result = result + M1.multiply(M2.T).multiply(M3)
    return result.tocsr()

In [14]:
RANDOM_SEED = 42
NUMBER_OF_NEGATIVES = 20000

def sample_negative_pairs(genes, diseases, positive_pairs, number_wanted, random_generator):
    # Random (gene, disease) pairs that are not positive, with no pair chosen twice.
    positive_set = set(positive_pairs)
    negatives = set()
    while len(negatives) < number_wanted:
        gene = genes[random_generator.integers(len(genes))]
        disease = diseases[random_generator.integers(len(diseases))]
        if (gene, disease) not in positive_set:
            negatives.add((gene, disease))
    return sorted(negatives)

random_generator = np.random.default_rng(RANDOM_SEED)
negative_pairs = sample_negative_pairs(nodes_by_class["Gene"], nodes_by_class["Disease"],
                                       target_pairs, NUMBER_OF_NEGATIVES, random_generator)
print(len(negative_pairs), "negative pairs, for example:", negative_pairs[:2])

20000 negative pairs, for example: [('A1BG', 'Hyperesthesia'), ('A1BG', 'Multiple_Sulfatase_Deficiency_Disease')]


In [15]:
def pair_positions(pairs, node_position):
    # The matrix row of each pair's gene and the matrix column of each pair's disease.
    rows = np.array([node_position["Gene"][gene] for gene, disease in pairs])
    columns = np.array([node_position["Disease"][disease] for gene, disease in pairs])
    return rows, columns

all_pairs = target_pairs + negative_pairs
labels = np.array([1] * len(target_pairs) + [0] * len(negative_pairs))
pair_rows, pair_columns = pair_positions(all_pairs, node_position)

print(f"{len(all_pairs):,} pairs: {labels.sum()} positive, {(labels == 0).sum():,} negative ({labels.mean():.2%} positive)")
pd.DataFrame({"gene": [p[0] for p in all_pairs[130:136]], "disease": [p[1] for p in all_pairs[130:136]], "label": labels[130:136]})

20,132 pairs: 132 positive, 20,000 negative (0.66% positive)


,gene,disease,label
0,WWOX,Neoplasms,1
1,ZMPSTE24,Neoplasms,1
2,A1BG,Hyperesthesia,0
3,A1BG,Multiple_Sulfatase_Deficiency_Disease,0
4,A2M,Canaliculitis,0
5,A2M,Erysipeloid,0


In [16]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

class MeanArcsinh(BaseEstimator, TransformerMixin):
    # Divide each column by its mean (learned from the training data), then apply arcsinh.
    def fit(self, features, labels=None):
        column_means = features.mean(axis=0)
        self.column_means_ = np.where(column_means > 0, column_means, 1.0)     # avoid dividing by zero
        return self

    def transform(self, features):
        return np.arcsinh(features / self.column_means_)

def make_ridge_model(C):
    # features -> mean-scaled arcsinh -> z-scores -> ridge logistic regression with balanced classes
    return make_pipeline(MeanArcsinh(), StandardScaler(),
                         LogisticRegression(C=C, class_weight="balanced", max_iter=3000))

In [17]:
def out_of_fold_scores(features, labels, C, random_seed):
    # The probability the model assigns to every pair when that pair was held out of training.
    scores = np.zeros(len(labels))
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)
    for train_positions, test_positions in folds.split(features, labels):
        model = make_ridge_model(C).fit(features[train_positions], labels[train_positions])
        scores[test_positions] = model.predict_proba(features[test_positions])[:, 1]
    return scores

In [18]:
def dwpc_features(steps_of_metapaths, w, adjacency_matrices, out_degree, in_degree, nodes_by_class, pair_rows, pair_columns):
    # One row per pair, one column per metapath: the DWPC of the pair for that metapath.
    weighted = weighted_adjacency_matrices(w, adjacency_matrices, out_degree, in_degree, nodes_by_class)
    columns = []
    for steps in steps_of_metapaths:
        dwpc_of_all_pairs = dwpc_matrix(steps, weighted)
        columns.append(np.asarray(dwpc_of_all_pairs[pair_rows, pair_columns]).ravel())
    return np.column_stack(columns)

In [19]:
short_metapath_names = [name for name, steps in metapaths.items() if len(steps) <= 2]
short_metapath_steps = [metapaths[name] for name in short_metapath_names]

path_counts = dwpc_features(short_metapath_steps, 0.0, adjacency_matrices, out_degree, in_degree,
                            nodes_by_class, pair_rows, pair_columns)
connects_enough_pairs = (path_counts > 0).sum(axis=0) >= 5

steps_for_choosing_w = [steps for steps, keep in zip(short_metapath_steps, connects_enough_pairs) if keep]
print(f"{len(steps_for_choosing_w)} of {len(short_metapath_steps)} short metapaths connect at least 5 pairs")

24 of 36 short metapaths connect at least 5 pairs


In [20]:
best_w = 0.2          # the damping exponent chosen in Section 7
best_C = 0.00003     # the regularisation strength chosen in Section 10

In [21]:
all_metapath_names = list(metapaths)
all_metapath_steps = [metapaths[name] for name in all_metapath_names]

dwpc_of_all_metapaths = dwpc_features(all_metapath_steps, best_w, adjacency_matrices, out_degree, in_degree,
                                      nodes_by_class, pair_rows, pair_columns)
path_counts_of_all_metapaths = dwpc_features(all_metapath_steps, 0.0, adjacency_matrices, out_degree, in_degree,
                                             nodes_by_class, pair_rows, pair_columns)

connects_enough_pairs = (path_counts_of_all_metapaths > 0).sum(axis=0) >= 5
metapath_names = [name for name, keep in zip(all_metapath_names, connects_enough_pairs) if keep]
metapath_features = dwpc_of_all_metapaths[:, connects_enough_pairs]          # DWPC at best_w
path_count_features = path_counts_of_all_metapaths[:, connects_enough_pairs]  # plain path count (w = 0)

print(f"{len(metapath_names)} of {len(all_metapath_names)} metapaths connect at least 5 pairs")
print("feature table shape (pairs x metapaths):", metapath_features.shape)

347 of 516 metapaths connect at least 5 pairs
feature table shape (pairs x metapaths): (20132, 347)


In [22]:
def degree_features(pairs, out_degree, in_degree):
    # For every metaedge touching Gene or Disease: the degree of each pair's gene / disease in it.
    columns, names = [], []
    for metaedge in out_degree:
        source_class, target_class = metaedge
        for role, position_in_pair in (("Gene", 0), ("Disease", 1)):
            nodes = [pair[position_in_pair] for pair in pairs]
            if source_class == role:
                columns.append([out_degree[metaedge][node] for node in nodes])
                names.append(f"{role}: out-degree in {source_class}>{target_class}")
            if target_class == role:
                columns.append([in_degree[metaedge][node] for node in nodes])
                names.append(f"{role}: in-degree in {source_class}>{target_class}")
    return np.array(columns, dtype=float).T, names

all_degree_features, all_degree_names = degree_features(all_pairs, out_degree, in_degree)
varies = all_degree_features.std(axis=0) > 0
degree_feature_table = all_degree_features[:, varies]
degree_names = [name for name, keep in zip(all_degree_names, varies) if keep]

print(len(degree_names), "degree features; table shape (pairs x features):", degree_feature_table.shape)
print("for example:", degree_names[:3])

37 degree features; table shape (pairs x features): (20132, 37)
for example: ['Disease: in-degree in Carbohydrate>Disease', 'Gene: in-degree in Carbohydrate>Gene', 'Disease: out-degree in Disease>Carbohydrate']


In [23]:
def build_adjacency_matrices(edges_by_metaedge, node_position, nodes_by_class):
    # One sparse 0/1 matrix per metaedge: rows = source nodes, columns = target nodes.
    matrices = {}
    for (source_class, target_class), edges in edges_by_metaedge.items():
        rows = [node_position[source_class][source] for source, target in edges]
        columns = [node_position[target_class][target] for source, target in edges]
        shape = (len(nodes_by_class[source_class]), len(nodes_by_class[target_class]))
        matrices[(source_class, target_class)] = sp.csr_matrix((np.ones(len(edges)), (rows, columns)), shape=shape)
    return matrices

rebuilt_matrices = build_adjacency_matrices(edges_by_metaedge, node_position, nodes_by_class)
assert all((rebuilt_matrices[metaedge] != adjacency_matrices[metaedge]).nnz == 0 for metaedge in adjacency_matrices)
print("check passed: the function reproduces the adjacency matrices of Section 5")

check passed: the function reproduces the adjacency matrices of Section 5


In [24]:
needed = ["hald_network", "metapaths", "adjacency_matrices", "out_degree", "in_degree", "node_position", "nodes_by_class",
          "target_pairs", "labels", "pair_rows", "pair_columns", "steps_for_choosing_w", "metapath_features",
          "metapath_names", "degree_feature_table", "dwpc_features", "out_of_fold_scores"]
assert all(name in globals() for name in needed), "the setup cells have not all been run"
print("setup complete: the main notebook's variables and functions are ready")

setup complete: the main notebook's variables and functions are ready


---
## Exercise 1: What does the damping exponent $w$ do?  *(7 minutes)*

### What you are asked to do
In Section 5 every path was given a weight that shrinks as the degrees of its nodes grow:

$$\mathrm{PDP}(\pi)=\prod_i \big[d^{\text{out}}\big]^{-w}\big[d^{\text{in}}\big]^{-w}.$$

The damping exponent $w$ controls how hard the hubs are punished: $w=0$ gives the plain path count, and a larger $w$ shrinks the weight of a path through a hub more. You will first work out the effect on one pair of paths by hand, and then measure how the choice of $w$ changes the model's performance.

### Part A: by hand
Two paths are exactly alike except that the first passes through a node of degree 4 and the second through a node of degree 400. To keep things simple, suppose that this node contributes a single factor $d^{-w}$ to each path, and that all the other degrees on the two paths are equal.

**Question 1.** By what factor is the weight of the second path smaller than the weight of the first path when $w=0$, when $w=0.5$ and when $w=1$?

### Part B: in code
Section 7 chose $w$ by computing the features of the 24 short metapaths (`steps_for_choosing_w`) for a grid of $w$ values, and scoring each set of features with the out-of-fold ridge model. The cell below repeats this for just three values of $w$. For each $w$ it computes the DWPC features (the function `dwpc_features` is the one from Section 7), obtains the out-of-fold scores, and records the AUROC and the average precision.

**Your task:** write the one line that computes `scores`. Use the function `out_of_fold_scores(features, labels, C, random_seed)` from Section 7, with `C=0.01` (the value used in Section 7) and `random_seed=RANDOM_SEED`. The cell takes about 30 seconds.

In [25]:
damping_exponents_to_try = [0, 0.5, 1.0]

damping_results = []
for w in damping_exponents_to_try:
    features = dwpc_features(steps_for_choosing_w, w, adjacency_matrices, out_degree, in_degree,
                             nodes_by_class, pair_rows, pair_columns)
    scores = ...          # TODO: the out-of-fold scores of the ridge model, with C=0.01 and random_seed=RANDOM_SEED
    damping_results.append((w, roc_auc_score(labels, scores), average_precision_score(labels, scores)))

damping_table = pd.DataFrame(damping_results, columns=["w", "AUROC", "average precision"]).set_index("w")
damping_table.round(4)

InvalidParameterError: The 'y_score' parameter of roc_auc_score must be an array-like. Got Ellipsis instead.

**Question 2.** How much does the AUROC change between $w=0$ and $w=0.5$, and between $w=0.5$ and $w=1$? What happens to the average precision as $w$ increases?

**Question 3.** The values in the table should agree with the corresponding rows of the table in Section 7. Why must they?

---
## Exercise 2: A DWPC by hand, then checked in code  *(8 minutes)*

### What you are asked to do
Compute a degree-weighted path count with pencil and paper on a graph small enough to see, and then ask the notebook's own functions to do the same calculation. If the two agree, you understand what the functions are doing.

### The toy graph
There are four genes (BRCA1, TP53, APOE, MLH1) and three diseases (BC, AD, T2D), with these edges:

| Gene → Disease | Disease → Disease |
|---|---|
| BRCA1 → BC | BC → AD |
| TP53 → BC | |
| TP53 → AD | |
| APOE → AD | |
| APOE → T2D | |
| MLH1 → BC | |
| MLH1 → T2D | |

We look at the metapath `Gene>Disease>Disease` (a gene, then a disease linked to it, then a disease linked to that one) with the damping exponent $w=0.5$. Remember that **degrees are counted within one metaedge**, so the first step uses the degrees in the metaedge (Gene, Disease) and the second step uses the degrees in the metaedge (Disease, Disease).

### Part A: by hand
1. Write the adjacency matrix $A$ of the metaedge (Gene, Disease), with the genes in the order BRCA1, TP53, APOE, MLH1 as rows and the diseases in the order BC, AD, T2D as columns. A cell is 1 if there is an edge.
2. Write down the out-degree of every gene and the in-degree of every disease in that metaedge (row sums and column sums of $A$).
3. The weight of an edge $x\to y$ is $W_{xy}=d^{\text{out}}(x)^{-w}\,d^{\text{in}}(y)^{-w}$. Compute $W_{\text{BRCA1},\text{BC}}$ and $W_{\text{MLH1},\text{BC}}$.
4. The only edge of the metaedge (Disease, Disease) is BC → AD. Compute its weight $W_{\text{BC},\text{AD}}$, using the degrees in *that* metaedge.
5. The weight of a path is the product of the weights of its edges. Compute the DWPC of the pair (BRCA1, AD) for this metapath: there is exactly one route, BRCA1 → BC → AD.
6. One gene has a DWPC of 0 to every disease for this metapath. Which one, and why?

### Part B: in code
The cell below is already written. It builds the adjacency matrices of the toy graph with the function `build_adjacency_matrices` of Section 9, counts the degrees with the same `Counter` code as Section 3, and prints the matrix and the degrees, so that you can check your answers to steps 1 and 2.

In [ ]:
toy_genes = ["BRCA1", "TP53", "APOE", "MLH1"]
toy_diseases = ["BC", "AD", "T2D"]
toy_edges_by_metaedge = {
    ("Gene", "Disease"): [("BRCA1", "BC"), ("TP53", "BC"), ("TP53", "AD"), ("APOE", "AD"),
                          ("APOE", "T2D"), ("MLH1", "BC"), ("MLH1", "T2D")],
    ("Disease", "Disease"): [("BC", "AD")],
}

toy_nodes_by_class = {"Gene": toy_genes, "Disease": toy_diseases}
toy_node_position = {class_name: {node: position for position, node in enumerate(names)}
                     for class_name, names in toy_nodes_by_class.items()}
toy_adjacency_matrices = build_adjacency_matrices(toy_edges_by_metaedge, toy_node_position, toy_nodes_by_class)

toy_out_degree, toy_in_degree = {}, {}
for metaedge, edges in toy_edges_by_metaedge.items():
    toy_out_degree[metaedge] = Counter(source for source, target in edges)
    toy_in_degree[metaedge] = Counter(target for source, target in edges)

print("adjacency matrix of (Gene, Disease), rows = genes, columns = diseases:")
print(pd.DataFrame(toy_adjacency_matrices[("Gene", "Disease")].toarray(), index=toy_genes, columns=toy_diseases).astype(int))
print("\nout-degrees of the genes:   ", dict(toy_out_degree[("Gene", "Disease")]))
print("in-degrees of the diseases: ", dict(toy_in_degree[("Gene", "Disease")]))

**Your task:** the notebook function `weighted_adjacency_matrices(w, adjacency_matrices, out_degree, in_degree, nodes_by_class)` (Section 5) turns the adjacency matrices into weighted matrices. Call it for the toy graph (use the `toy_` variables above) and store the result in `toy_weighted_matrices`. The cell then uses `dwpc_matrix` to compute the DWPC of every (gene, disease) pair. Is the entry for (BRCA1, AD) the number you found in step 5?

In [ ]:
toy_w = 0.5
toy_weighted_matrices = ...      # TODO: the weighted matrices of the toy graph, using weighted_adjacency_matrices

toy_dwpc = dwpc_matrix(metapaths["Gene>Disease>Disease"], toy_weighted_matrices).toarray()
print("DWPC of Gene>Disease>Disease, rows = genes, columns = diseases:")
print(pd.DataFrame(toy_dwpc, index=toy_genes, columns=toy_diseases).round(3))

**Question 4.** Without MLH1 (and its edges) BRCA1's DWPC to AD would be $1^{-1/2}\cdot 2^{-1/2}\approx 0.707$. With MLH1 it is 0.577, although none of BRCA1's own edges changed. Why?

---
## Exercise 3: An ROC curve by hand  *(8 minutes)*

### What you are asked to do
Build an ROC curve and its area from a handful of scores, with no software, and then confirm the result in code. This is the calculation that Section 11 performs on 20,132 pairs.

### The set-up
A model has scored six gene–disease pairs. Three of them are truly positive (label 1) and three are not (label 0):

| score | 0.9 | 0.8 | 0.7 | 0.6 | 0.4 | 0.3 |
|---|---|---|---|---|---|---|
| true label | 1 | 1 | 0 | 1 | 0 | 0 |

For a **threshold** $\tau$, every pair with a score of at least $\tau$ is *predicted* positive. Two rates describe the result:

$$\mathrm{TPR}=\frac{TP}{\text{number of positives}},\qquad \mathrm{FPR}=\frac{FP}{\text{number of negatives}},$$

where $TP$ is the number of real positives that are predicted positive and $FP$ is the number of negatives that are predicted positive. The **ROC curve** plots TPR (vertical) against FPR (horizontal) as the threshold is lowered from above the highest score to below the lowest.

### Part A: by hand
1. For each of the thresholds $\tau=0.9,\ 0.8,\ 0.7,\ 0.6,\ 0.4,\ 0.3$ fill in a table with the columns TP, FP, TPR and FPR.
2. Plot the points (FPR, TPR) on squared paper, starting from (0, 0), and join them with straight lines.
3. The **AUROC** is the probability that a randomly chosen positive pair is scored higher than a randomly chosen negative pair. There are $3\times 3=9$ pairs made of one positive and one negative. In how many of them is the positive scored higher? Write the AUROC as that fraction.

### Part B: in code
The cell below is already written. It computes the AUROC with `roc_auc_score` and lists the points of the curve with `roc_curve`. Compare them with your table and with your fraction.

In [ ]:
from sklearn.metrics import roc_curve

toy_scores = np.array([0.9, 0.8, 0.7, 0.6, 0.4, 0.3])
toy_labels = np.array([1,   1,   0,   1,   0,   0])

print("AUROC =", round(roc_auc_score(toy_labels, toy_scores), 3))

false_positive_rate, true_positive_rate, thresholds = roc_curve(toy_labels, toy_scores, drop_intermediate=False)
pd.DataFrame({"threshold": thresholds, "FPR": false_positive_rate.round(2), "TPR": true_positive_rate.round(2)})

**Your task:** make a copy of `toy_scores` in which the negative pair that was scored 0.7 is scored 0.5 instead (it is the third pair), store it in `changed_scores`, and run the cell to see the new AUROC.

In [ ]:
changed_scores = ...      # TODO: a copy of toy_scores in which the negative scored 0.7 is scored 0.5 instead
print("AUROC with the changed score =", round(roc_auc_score(toy_labels, changed_scores), 3))

**Question 5.** What is the AUROC with the changed score, and why does it have that value? Check it with a 9-pair count like the one in step 3.

**Question 6 (think, no calculation).** In the HALD data there are 132 positive and 20,000 negative pairs. A model can have an AUROC close to 1 and still be *wrong about most of the pairs it flags* when we allow a false positive rate of 1%. Why?

---
## Exercise 4: Testing against fairer negative examples  *(7 minutes)*

### What you are asked to do
In Section 6 we chose the negative examples at random, and found that the positive genes and diseases are far better connected than random ones. A model can therefore score well simply by recognising well-connected nodes (hubs). In this exercise you build **degree-matched negatives**, which are as well connected as the positives, and see how much of the performance survives.

### The idea
For every positive pair (gene $g$, disease $d$) we draw 20 negative pairs whose gene is in the **same degree decile** as $g$ and whose disease is in the same degree decile as $d$. A *decile* is one tenth of the nodes of a class when they are sorted by their total degree (decile 0 is the least connected tenth, decile 9 the most connected tenth). The *total degree* of a node is the number of edges of any kind that leave or arrive at it.

### Step 1: group the nodes by degree (already written)
Read the cell and run it. The function `total_degree_of_every_node` adds up the degrees of a node over all metaedges, and `degree_decile_of_every_node` splits the nodes of one class into ten groups of equal size. The cell then prints in which decile the genes of the 132 positive pairs fall.

In [ ]:
def total_degree_of_every_node(out_degree, in_degree):
    # The number of edges of any kind that leave or arrive at each node (node names are unique across classes).
    total_degree = Counter()
    for degrees in list(out_degree.values()) + list(in_degree.values()):
        total_degree.update(degrees)
    return total_degree

def degree_decile_of_every_node(nodes, total_degree):
    # Group nodes into ten groups of equal size by total degree: 0 = least connected, 9 = most connected.
    degrees = pd.Series({node: total_degree[node] for node in nodes})
    deciles = pd.qcut(degrees.rank(method="first"), 10, labels=False)
    return deciles.to_dict()

total_degree = total_degree_of_every_node(out_degree, in_degree)
gene_decile = degree_decile_of_every_node(nodes_by_class["Gene"], total_degree)
disease_decile = degree_decile_of_every_node(nodes_by_class["Disease"], total_degree)

genes_in_decile = {decile: [gene for gene, d in gene_decile.items() if d == decile] for decile in range(10)}
diseases_in_decile = {decile: [disease for disease, d in disease_decile.items() if d == decile] for decile in range(10)}

positive_gene_deciles = pd.Series([gene_decile[gene] for gene, disease in target_pairs])
print("degree decile of the genes in the 132 positive pairs (0 = least connected, 9 = most connected):")
print(positive_gene_deciles.value_counts().sort_index().to_string())

### Step 2: sample the matched negatives
The function below draws the negatives. For each positive pair it looks up the deciles of its gene and disease, and keeps drawing a random gene from the same decile and a random disease from the same decile until it has 20 pairs that are neither positives nor already chosen.

**Your task:** write the two lines that draw `gene` and `disease`. A random element of a list `candidates` is `candidates[random_generator.integers(len(candidates))]`, the same device that `sample_negative_pairs` used in Section 6.

In [ ]:
def sample_matched_negatives(positive_pairs, gene_decile, disease_decile, genes_in_decile, diseases_in_decile,
                             negatives_per_positive, random_generator):
    # For every positive pair, random pairs whose gene and disease are in the same degree deciles as its own.
    positive_set = set(positive_pairs)
    negatives = set()
    for positive_gene, positive_disease in positive_pairs:
        chosen = set()
        while len(chosen) < negatives_per_positive:
            candidate_genes = genes_in_decile[gene_decile[positive_gene]]
            candidate_diseases = diseases_in_decile[disease_decile[positive_disease]]
            gene = ...          # TODO: a gene chosen at random from candidate_genes
            disease = ...       # TODO: a disease chosen at random from candidate_diseases
            if (gene, disease) not in positive_set and (gene, disease) not in negatives:
                chosen.add((gene, disease))
        negatives |= chosen
    return sorted(negatives)

random_generator = np.random.default_rng(7)
matched_negative_pairs = sample_matched_negatives(target_pairs, gene_decile, disease_decile, genes_in_decile,
                                                  diseases_in_decile, 20, random_generator)
print(len(matched_negative_pairs), "matched negative pairs, for example:", matched_negative_pairs[:2])

### Step 3: build the features of the matched data set (already written)
Run the cell. It puts the positive pairs and their matched negatives into one data set, with labels, and computes for it the same two kinds of features as in Section 8: the DWPC of the 347 kept metapaths at `best_w`, and the degree features (using the same columns, `varies`, that Section 8 kept). It takes about 20 seconds.

In [ ]:
matched_pairs = target_pairs + matched_negative_pairs
matched_labels = np.array([1] * len(target_pairs) + [0] * len(matched_negative_pairs))
matched_rows, matched_columns = pair_positions(matched_pairs, node_position)

kept_metapath_steps = [metapaths[name] for name in metapath_names]       # the 347 metapaths kept in Section 8
matched_metapath_features = dwpc_features(kept_metapath_steps, best_w, adjacency_matrices, out_degree, in_degree,
                                          nodes_by_class, matched_rows, matched_columns)
matched_degree_features = degree_features(matched_pairs, out_degree, in_degree)[0][:, varies]

print(f"{len(matched_pairs):,} pairs: {matched_labels.sum()} positive, {(matched_labels == 0).sum():,} negative ({matched_labels.mean():.1%} positive)")

### Step 4: compare random and matched negatives
Now score three feature sets with the same ridge model, once on the data set of Section 6 (random negatives) and once on the matched data set. **Your task:** write the line that computes `matched_scores`, in the same way as the line above it but with `matched_features` and `matched_labels`. The cell reports the AUROC for both data sets and, for the matched data set, the average precision. (The average precision of the two data sets cannot be compared, because 0.66% of the first and 4.8% of the second are positive.) The cell takes about a minute.

In [ ]:
feature_sets_to_compare = {
    "degree features only": (degree_feature_table, matched_degree_features),
    "DWPC":                 (metapath_features, matched_metapath_features),
    "DWPC + degree":        (np.hstack([metapath_features, degree_feature_table]),
                             np.hstack([matched_metapath_features, matched_degree_features])),
}

rows = []
for name, (random_features, matched_features) in feature_sets_to_compare.items():
    random_scores = out_of_fold_scores(random_features, labels, best_C, RANDOM_SEED)
    matched_scores = ...      # TODO: the same, for matched_features and matched_labels
    rows.append((name, roc_auc_score(labels, random_scores), roc_auc_score(matched_labels, matched_scores),
                 average_precision_score(matched_labels, matched_scores)))

pd.DataFrame(rows, columns=["features", "AUROC, random negatives", "AUROC, matched negatives",
                            "average precision, matched negatives"]).set_index("features").round(3)

**Question 7.** What happens to the AUROC of the degree-only model when the negatives are degree-matched? Why does it not fall to 0.5?

**Question 8.** With the matched negatives, do the metapath features add anything to the degree features? Look at both the AUROC and the average precision.

**Question 9.** In one or two sentences, why is the matched data set a fairer test of the method than the random one?